In [3]:
# Fix reconstruction: use outer(vec(A_k), vec(B_k)) (NOT kron) to form M_k.
import numpy as np
from typing import List, Dict, Any

def schmidt_two_fragments_ci(
    ci: np.ndarray,
    norb_f: List[int],
) -> Dict[str, Any]:
    assert len(norb_f) == 2, "norb_f must have length 2 for two fragments."
    d1, d2 = 2**norb_f[0], 2**norb_f[1]
    D = d1 * d2
    assert ci.shape == (D, D), f"ci must be {(D, D)} for dims ({d1},{d2})."

    # ci -> T(Aα,Bα,Aβ,Bβ)
    T = ci.reshape(d1, d2, d1, d2, order="C")
    # group (Aα,Aβ)|(Bα,Bβ)
    T_perm = np.transpose(T, (0, 2, 1, 3))              # (d1,d1,d2,d2)
    M = T_perm.reshape(d1*d1, d2*d2, order="C")         # (d1*d1, d2*d2)

    # SVD
    U, s, Vh = np.linalg.svd(M, full_matrices=False)
    V = Vh.conj().T

    # Per-fragment CI matrices
    A_list, B_list = [], []
    for k in range(len(s)):
        A_k = U[:, k].reshape(d1, d1, order="C")
        B_k = np.conjugate(V[:, k]).reshape(d2, d2, order="C")
        A_list.append(A_k)
        B_list.append(B_k)

    # Reconstruct via outer(vec(A_k), vec(B_k))
    def _term_ci_from_AB(A_k, B_k, s_k):
        Mk = np.outer(A_k.reshape(-1, order="C"),
                      B_k.reshape(-1, order="C"))        # (d1*d1, d2*d2)
        Mk *= s_k
        Tperm_k = Mk.reshape(d1, d1, d2, d2, order="C")  # (d1,d1,d2,d2)
        Tk = np.transpose(Tperm_k, (0, 2, 1, 3))         # (d1,d2,d1,d2)
        return Tk.reshape(D, D, order="C")

    term_ci_list = [_term_ci_from_AB(A_list[k], B_list[k], s[k]) for k in range(len(s))]
    reconstruction = np.sum(term_ci_list, axis=0) if term_ci_list else np.zeros_like(ci)
    err = np.linalg.norm(reconstruction - ci)

    return {
        "s": s,
        "A_list": A_list,
        "B_list": B_list,
        "term_ci_list": term_ci_list,
        "reconstruction": reconstruction,
        "recon_error_fro": float(err),
    }


# Re-run the demo with the corrected reconstruction
def _demo_two_fragments_fixed():
    norb_f = [1, 2]
    d1, d2, D = 2**norb_f[0], 2**norb_f[1], 2**sum(norb_f)

    rng = np.random.default_rng(0)
    # build random complex sum of K product terms in the (Aα,Aβ | Bα,Bβ) grouped representation
    K = 3
    s_true = rng.random(K) + 0.1
    A_true = [rng.normal(size=(d1, d1)) + 1j*rng.normal(size=(d1, d1)) for _ in range(K)]
    B_true = [rng.normal(size=(d2, d2)) + 1j*rng.normal(size=(d2, d2)) for _ in range(K)]

    def _unperm(A, B, s_k):
        M = np.outer(A.reshape(-1), B.reshape(-1)) * s_k    # (d1*d1, d2*d2)
        Tperm = M.reshape(d1, d1, d2, d2)
        T = np.transpose(Tperm, (0, 2, 1, 3)).reshape(D, D)
        return T

    ci = sum(_unperm(A_true[k], B_true[k], s_true[k]) for k in range(K))
    print(ci)

    out = schmidt_two_fragments_ci(ci, norb_f)
    print(out["A_list"])
    return {
        "dims": {"d1": d1, "d2": d2, "D": D},
        "num_sv": len(out["s"]),
        "first_three_sv": out["s"][:3],
        "fro_error": out["recon_error_fro"],
        "A0_shape": out["A_list"][0].shape,
        "B0_shape": out["B_list"][0].shape,
        "term0_ci_shape": out["term_ci_list"][0].shape,
    }

_demo_two_fragments_fixed()


[[-0.78363201-0.25464282j -0.35786806-0.54310336j  0.34421012+0.48080842j
  -0.99029815-1.13530504j  0.96461085+1.62730141j  0.85437411+0.7752363j
   0.90832562-0.65943807j -0.26128891-0.40185234j]
 [-0.7035483 +0.23760266j -1.1862933 -0.06119881j -1.14755988+0.54018676j
  -0.54239619-0.22153691j  1.89597611-2.15775291j  0.75906028-0.83506111j
   0.54828113-0.2814569j  -1.40090803-0.42988667j]
 [ 0.42715088-0.40048634j -0.38026903-0.41397141j -0.22170957+0.72234669j
   1.05204384+0.55823755j -1.99977407+0.80033057j -0.73597236+1.35016395j
   2.30259761-0.17756403j -1.00991866-0.44433787j]
 [-0.01495414+1.02551503j -0.49707526-0.96992425j -0.15083829+1.36420562j
   0.7719966 +0.325843j   -0.84080108-2.33907325j -0.41119728+1.02693888j
   0.55023622-2.56618438j -2.76647183-0.44235565j]
 [ 0.67715182+1.39249812j  0.02903461+0.16270929j -0.17686564-0.16314774j
   1.01149536+1.01392702j -0.10359517+1.4585882j  -0.21189277+0.95910464j
   0.57350334-0.97082377j -1.19374761+2.13542298j]
 [ 2.2

{'dims': {'d1': 2, 'd2': 4, 'D': 8},
 'num_sv': 4,
 'first_three_sv': array([10.58247441,  5.03524976,  1.48696256]),
 'fro_error': 9.491188164432655e-15,
 'A0_shape': (2, 2),
 'B0_shape': (4, 4),
 'term0_ci_shape': (8, 8)}

In [1]:
import numpy as np
from pyscf import gto, scf, mcscf, ao2mo

# 1) System + mean-field
mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g', spin=0, charge=0)
mf = scf.RHF(mol).run()

# 2) Set up CASCI
ncas, nelecas = 2, 2  # example: 2 active orbitals, 2 active electrons
mc = mcscf.CASCI(mf, ncas, nelecas)

# pick/prepare your active-space MO coeffs (use your own if you already have them)
mo = mcscf.addons.sort_mo_by_irrep(mc, mf.mo_coeff, {'A1g': [0,1]}) if hasattr(mcscf.addons, 'sort_mo_by_irrep') else mf.mo_coeff
mc.mo_coeff = mo

# 3) Build active-space one-/two-electron integrals and core energy
h1e, ecore = mc.h1e_for_cas(mc.mo_coeff)
mo_cas = mc.mo_coeff[:, :ncas]
eri_cas = ao2mo.incore.full(mf._eri, mo_cas, compact=False).reshape(ncas, ncas, ncas, ncas)

# 4) Prepare your CI vector (must match fcisolver and (ncas, nelecas))
# For the default spin-adapted direct_spin1 solver, `ci` is a 2D array with shapes
# (Na_det, Nb_det). Here we just make a normalized random example:
na = nelecas//2
nb = nelecas - na
na_dim = mc.fcisolver.states_count(ncas, na)
nb_dim = mc.fcisolver.states_count(ncas, nb)
ci = np.random.randn(na_dim, nb_dim)
ci /= np.linalg.norm(ci)  # normalize your CI!

# 5) Evaluate energy of THIS CI (no diagonalization/optimization)
E = mc.fcisolver.energy(h1e, eri_cas, ncas, nelecas, ci, ecore=ecore)
print("Energy for supplied CI:", E)


converged SCF energy = -1.11675930739643


TypeError: object of type 'NoneType' has no len()